# Part B — Neural Style Transfer

**Group 7 | Office-Home | Art (content) → Clipart (style)**

Implementation of Gatys et al. (2016):
- **Content layer**: `relu4_2` (VGG-19 index 22) — captures object structure
- **Style layers**: `relu1_1, relu2_1, relu3_1, relu4_1, relu5_1` — textures at multiple scales
- **Optimiser**: L-BFGS on pixel values of the generated image

Deliverable: ≥30 style-transferred images per class (180+ total) saved to `data/synthetic_target/`.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import numpy as np

from src.utils import get_device, CLASSES, DATA_ROOT
from src.style_transfer import (
    style_transfer, generate_synthetic_dataset,
    load_image, tensor_to_pil
)

device = get_device()
print(f'Device: {device}')

SYNTHETIC_DIR = '../data/synthetic_target'

## 1. Single example — visualise style transfer for one image

In [ ]:
# Pick one content (Art) and one style (Clipart) image
cls = 'Laptop'
art_dir     = os.path.join(DATA_ROOT, 'Art', cls)
clipart_dir = os.path.join(DATA_ROOT, 'Clipart', cls)

content_path = sorted(os.listdir(art_dir))[0]
style_path   = sorted(os.listdir(clipart_dir))[0]
content_path = os.path.join(art_dir, content_path)
style_path   = os.path.join(clipart_dir, style_path)

print(f'Content: {content_path}')
print(f'Style:   {style_path}')

# Run NST (alpha/beta ratio: try 1e-4 and 1e-3 — adjust to taste)
generated = style_transfer(
    content_path, style_path,
    device=device,
    alpha=1.0, beta=1e4,
    n_steps=300,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(Image.open(content_path)); axes[0].set_title('Content (Art)')
axes[1].imshow(Image.open(style_path));   axes[1].set_title('Style (Clipart)')
axes[2].imshow(generated);                axes[2].set_title('Generated')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('../figures/nst_single_example.png', dpi=150)
plt.show()

## 2. Hyperparameter exploration — α/β ratio

Try three ratios to understand the content-style tradeoff:
- `α/β = 1e-3` — strong style, may distort content
- `α/β = 1e-4` — balanced
- `α/β = 1e-5` — subtle style

In [ ]:
ratios = [
    (1.0, 1e3, '1e-3'),
    (1.0, 1e4, '1e-4'),
    (1.0, 1e5, '1e-5'),
]

fig, axes = plt.subplots(1, len(ratios) + 1, figsize=(18, 5))
axes[0].imshow(Image.open(content_path)); axes[0].set_title('Content (Art)'); axes[0].axis('off')

for i, (alpha, beta, label) in enumerate(ratios):
    gen = style_transfer(
        content_path, style_path, device=device,
        alpha=alpha, beta=beta, n_steps=200
    )
    axes[i + 1].imshow(gen)
    axes[i + 1].set_title(f'α/β = {label}')
    axes[i + 1].axis('off')

plt.suptitle('Effect of α/β ratio on style transfer', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/nst_ratio_comparison.png', dpi=150)
plt.show()

## 3. Figure B — Gallery: one example per class (required figure)

Content image | Style image | Generated image — for each of the 6 classes.

In [ ]:
fig = plt.figure(figsize=(15, 4 * len(CLASSES)))
gs  = gridspec.GridSpec(len(CLASSES), 3, figure=fig)

for row, cls in enumerate(CLASSES):
    art_dir     = os.path.join(DATA_ROOT, 'Art', cls)
    clipart_dir = os.path.join(DATA_ROOT, 'Clipart', cls)

    c_path = os.path.join(art_dir,     sorted(os.listdir(art_dir))[0])
    s_path = os.path.join(clipart_dir, sorted(os.listdir(clipart_dir))[0])

    gen = style_transfer(c_path, s_path, device=device, alpha=1.0, beta=1e4, n_steps=300)

    for col, (img, title) in enumerate([
        (Image.open(c_path), f'{cls}\nContent (Art)'),
        (Image.open(s_path), 'Style (Clipart)'),
        (gen,                'Generated'),
    ]):
        ax = fig.add_subplot(gs[row, col])
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis('off')

plt.suptitle('Neural Style Transfer Gallery — One example per class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../figures/part_b_gallery.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/part_b_gallery.png')

## 4. Generate full synthetic dataset (180+ images)

This cell generates ≥30 style-transferred images per class → 180+ total.  
**Expected time**: 1-3 min/image on CPU → plan for overnight run; ~10× faster on GPU/MPS.

Already-generated images are skipped automatically.

In [ ]:
# Choose α/β based on the visual quality assessment above
ALPHA = 1.0
BETA  = 1e4   # adjust: 1e3 for stronger style, 1e5 for subtle
N_STEPS = 300

generate_synthetic_dataset(
    output_dir=SYNTHETIC_DIR,
    n_per_class=30,
    alpha=ALPHA,
    beta=BETA,
    n_steps=N_STEPS,
)

# Count generated images
for cls in CLASSES:
    cls_dir = os.path.join(SYNTHETIC_DIR, cls)
    n = len([f for f in os.listdir(cls_dir) if f.endswith('.jpg')]) if os.path.isdir(cls_dir) else 0
    print(f'  {cls}: {n} synthetic images')